The goal of this notebook is to get the time delay between the (bugged timestamps in the) kinect stream and the other streams in the .xdf file. 

# The problem

Due to a bug in LSL_Kinect, the timestamps in the kinect streams are relative to the computer's startup time, whereas timestamps should be defined by LSL to enable synchronization with other streams.  
As a consequence, the kinect streams are delayed compared to the other streams, and the value of the delay is unknown but (supposed) constant.

This is an example of what we get : 

```
LSL-time = xdf_mouse_marker_time  :       21786.679s to       22332.342s,     48 samples for a duration of  545.662s
LSL-time = xdf_mouse_mocap_time   :       21806.683s to       22310.975s,     12 samples for a duration of  504.292s
BUG-time = xdf_kinect_marker_time :         170.238s to         839.806s,     11 samples for a duration of  669.569s
BUG-time = xdf_kinect_mocap_time  :         246.284s to         839.798s,  15667 samples for a duration of  593.514s
CSV-time = csv_kinect_mocap_time  :  1616421533.460s to  1616422126.974s,  15667 samples for a duration of  593.514s
``` 

*NOTE: the BUG has been corrected on 31 July 2023, for LSL_Kinect versions >= 1.2.0.*

# The solution
Fortunately, the kinect streams are also saved in .csv files, and the mouse streams are saved in other .csv files.

For each device, we have two streams: 
- the mocap stream, also saved in a .csv file 
- the marker stream, also saved in a .csv file 

Hence, we have :

- in the .xdf file : 
    - the kinect streams: with buggy timestamps (i.e., relative to the computer's startup time, which is unknown)
    - the mouse streams: with LSL timestamps (i.e., in sync with the other streams)

- in the .csv files :
    - the mouse streams, with timestamps as current time in milliseconds (e.g., 1616422003021 a [java currentTimeMillis](https://docs.oracle.com/javase/8/docs/api/java/lang/System.html#currentTimeMillis--))
    - the kinect streams, with timestamps as current time in milliseconds (e.g., '2021-03-22 15:06:43.021' a date-time string, corresponding to 1616422003021)

Stated differently, we have 3 time references:
- `LSL-kinect-time`: the timestamps of the Kinect streams (bugged)
- `LSL-time`: the LSL timestamps (in sync with the other streams)
- `CSV-time`: the computer's current time in JAVA milliseconds (the time reference of all .csv files)

To know the relations between the 3 time references, one solution is to compute:
- **kinect-to-csv delay**: the delay between the kinect streams and the corresponding csv files
- **mouse-to-csv delay**: the delay between mouse streams and the corresponding csv files
- **kinect-to-mouse delay**: the delay between the kinect streams and the mouse streams

From the previous, we can compute the corrected kinect timestamps : the timestamps of the kinect stream shifted by the kinect-to-mouse delay


NOTE: if the kinect and mouse are registered on the same computer, the current time is the same for both csv files. If this is not the case, we would need to take into account the time difference between the two computers clocks (which is unknown, and is the reason we use LSL...). The good news is that ReArm registrations (normally) use the same computer for running LSL-mouse and LSL-kinect. 



# The action plan, step by step

## Computing the mouse-to-csv delay
The mouse-to-csv delay is the delay between the mouse streams and the mouse csv files.

As only the mouse markers csv files are systematically saved in the ReArm data set, we  will compare the timestamps from:  
- `LSL-time` in the mouse marker stream
- `CSV-time` in the mouse markers csv file

Do do so, we need to:
- read the the `LSL-time` from the mouse marker stream 
- select the corresponding csv file (if it exists)
- read the csv file to extract the `CSV-time` column
- check that we have the same number of rows in the csv file and in the LSL stream (it should be the case)
- compare timestamps, after converting `CSV-time` to seconds
- get the mean and std of the delay

## Computing the kinect-to-csv delay
The kinect-to-csv delay is the delay between the kinect streams and the kinect csv files.

Here, we take advantage of some very good news: the first column of the kinect mocap stream contains the CSV-time. Therefore, we can simply compare the timestamps of :  
- `LSL-kinect-time` in the kinect mocap stream 
- `CSV-time` in the first column of the kinect mocap stream 

Do do so, we merely need to:
- read the the `LSL-kinect-time` from the kinect mocap stream 
- read the first column of the kinect mocap stream to extract the `CSV-time` column
- compare timestamps, after converting `CSV-time` to seconds
- get the mean and std of the delay


## Computing the kinect-to-mouse delay
The kinect-to-mouse delay is the delay between the kinect streams and the mouse streams.

To do so, we simply go from the kinect -> to csv -> to mouse time references:  
- kinect_to_mouse_delay_s = kinect_to_csv_delay_s - mouse_to_csv_delay_s

## Correcting the kinect timestamps
The corrected kinect timestamps are the timestamps of the kinect stream shifted by the kinect-to-mouse delay: 
- corrected_kinect_timestamp = LSL-kinect-time + kinect_to_mouse_delay_s



# The code for the solution

## Imports

In [ ]:
import pyxdf
import numpy as np
import tempfile
import os
import difflib
import logging
from PIL import Image

# for the tests
import matplotlib.pyplot as plt

In [ ]:
# import the mouse and kinect csv files utilities
run_command = "-i check_csv_files.ipynb"
if "doRunTests" in globals():
    if doRunTests == False:
        get_ipython().run_line_magic("run", run_command)
    else:
        doRunTests = False
        get_ipython().run_line_magic("run", run_command)
        doRunTests = True

if "doRunTests" not in globals():
    doRunTests = False  # necessary to %run
    get_ipython().run_line_magic("run", run_command)
    del doRunTests  # remove the variable from the namespace

## Global initialization

In [ ]:
# this should be the set to false for production use
# doRunTests = False
# do_debug = False

# and set to true for testing (default) but can be already set to false from outside (e.g., by the test script)
if "doRunTests" not in globals():
    doRunTests = True
    do_debug = True

if doRunTests:
    # the best for debug-test plots (external window that you can make fullscreen and zoom)
    %matplotlib qt 
    
    # the best for inline plots (in the notebook)
    # %matplotlib inline

## Kinect-to-csv delay

The kinect-to-csv delay is easy to compute: it is the time difference between the `LSL-kinect-time` and the `TimeSpan` column in the kinect mocap stream.

### Read the mouse and kinect streams from the .xdf file

NOTE: For the data recorded with the event IDE software (after patient 23 os so), the mouse streams are not present for the reaching task.

In [ ]:
def read_xdf_mouse_kinect(xdf_fullFname):
    """Read the XDF file and return a dict containing :

    - "xdf_mouse_marker_time": np.array 1D
    - "xdf_mouse_marker_data": list of makers [str]
    - "xdf_NIC_Quality_time": np.array 1D
    - "xdf_mouse_mocap_time": np.array 1D
    - "xdf_kinect_marker_time": np.array 1D
    - "xdf_kinect_mocap_time": np.array 1D
    - "csv_kinect_mocap_time": np.array 1D

    """

    if 1 == 2:
        # explore what is in the XDF file
        xdf_data, header = pyxdf.load_xdf(
            filename=xdf_fullFname,
            synchronize_clocks=True,
            dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
            verbose=False,
        )

        # print the stream names and types
        for stream in xdf_data:
            print(stream["info"]["name"][0], stream["info"]["type"][0])

    # NOTE: do not forget to synchronize the clocks for all streams
    xdf_data, header = pyxdf.load_xdf(
        filename=xdf_fullFname,
        select_streams=[
            {"type": "MoCap", "name": "EuroMov-Mocap-Kinect"},
            {"type": "MoCap", "name": "Mouse"},
            {"type": "MoCap", "name": "MouseData"},  # for the new files
            {"type": "Markers", "name": "EuroMov-Markers-Kinect"},
            {"type": "Markers", "name": "Mouse"},
            {"type": "Markers", "name": "MouseMarkers"},
            # {"type": "Quality", "name": "NIC-Quality"},
            {"type": "Quality"},  # NIC changed the name
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
        verbose=False,
    )

    # get the streams
    kinect_mocap = [
        stream
        for stream in xdf_data
        if stream["info"]["name"][0] == "EuroMov-Mocap-Kinect"
    ]

    kinect_markers = [
        stream
        for stream in xdf_data
        if stream["info"]["name"][0] == "EuroMov-Markers-Kinect"
    ]

    mouse_markers = [
        xdf_stream
        for xdf_stream in xdf_data
        if "Mouse" in xdf_stream["info"]["name"][0]
        and xdf_stream["info"]["type"][0] == "Markers"
    ]

    mouse_mocap = [
        xdf_stream
        for xdf_stream in xdf_data
        if xdf_stream["info"]["name"][0] == "Mouse"
        or xdf_stream["info"]["name"][0] == "MouseData"
        and xdf_stream["info"]["type"][0] == "MoCap"
    ]

    NIC_Quality = [
        xdf_stream
        for xdf_stream in xdf_data
        if xdf_stream["info"]["type"][0] == "Quality"
    ]

    # Initialize the return values
    xdf_mouse_marker_time = np.array([-1.0])
    xdf_mouse_marker_data = []
    xdf_mouse_mocap_time = np.array([-1.0])
    xdf_mouse_mocap_data = []
    xdf_kinect_marker_time = np.array([-1.0])
    xdf_kinect_marker_data = []
    xdf_kinect_mocap_time = np.array([-1.0])
    xdf_kinect_mocap_data = []

    # def get_time_stamps(stream):
    #     if stream:
    #         return stream[0]["time_stamps"]
    #     return np.array([-1.0])

    if kinect_mocap:
        kinect_mocap = kinect_mocap[0]
        xdf_kinect_mocap_time = kinect_mocap["time_stamps"]
        xdf_kinect_mocap_data = kinect_mocap["time_series"]
        # the time stamps (same in CSV file) are in the first column of the time_series
        csv_kinect_mocap_time = kinect_mocap["time_series"][:, 0] / 1000.0

    if kinect_markers:
        kinect_markers = kinect_markers[0]
        xdf_kinect_marker_time = kinect_markers["time_stamps"]
        xdf_kinect_marker_data = kinect_markers["time_series"]

    if mouse_markers:
        mouse_markers = mouse_markers[0]
        xdf_mouse_marker_time = mouse_markers["time_stamps"]
        xdf_mouse_marker_data = mouse_markers["time_series"]

    if mouse_mocap:
        mouse_mocap = mouse_mocap[0]
        xdf_mouse_mocap_time = mouse_mocap["time_stamps"]
        xdf_mouse_mocap_data = mouse_mocap["time_series"]

    if NIC_Quality:
        NIC_Quality = NIC_Quality[0]
        xdf_NIC_Quality_time = NIC_Quality["time_stamps"]

    return {
        "xdf_mouse_marker_time": xdf_mouse_marker_time,
        "xdf_mouse_marker_data": xdf_mouse_marker_data,
        "xdf_mouse_mocap_time": xdf_mouse_mocap_time,
        "xdf_mouse_mocap_data": xdf_mouse_mocap_data,
        "xdf_kinect_marker_time": xdf_kinect_marker_time,
        "xdf_kinect_marker_data": xdf_kinect_marker_data,
        "xdf_kinect_mocap_time": xdf_kinect_mocap_time,
        "xdf_kinect_mocap_data": xdf_kinect_mocap_data,
        "csv_kinect_mocap_time": csv_kinect_mocap_time,
        "xdf_NIC_Quality_time": xdf_NIC_Quality_time,
    }


if doRunTests:

    def test_read_xdf_mouse_kinect():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Reaching/ReArm_C1P42_20240603_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_c.xdf"

        xdf_mouse_kinect = read_xdf_mouse_kinect(xdf_fullFname)
        xdf_mouse_marker_time = xdf_mouse_kinect["xdf_mouse_marker_time"]
        xdf_mouse_mocap_time = xdf_mouse_kinect["xdf_mouse_mocap_time"]
        xdf_NIC_Quality_time = xdf_mouse_kinect["xdf_NIC_Quality_time"]
        xdf_kinect_marker_time = xdf_mouse_kinect["xdf_kinect_marker_time"]
        xdf_kinect_mocap_time = xdf_mouse_kinect["xdf_kinect_mocap_time"]
        csv_kinect_mocap_time = xdf_mouse_kinect["csv_kinect_mocap_time"]

        def print_stats(name, data):
            print(
                f"{name}: {data[0]:15.3f}s to {data[-1]:15.3f}s, {len(data):6.0f} samples for a duration of {data[-1] - data[0]:8.3f}s"
            )

        print("----")
        print_stats("LSL-time = xdf_mouse_marker_time  ", xdf_mouse_marker_time)
        print_stats("LSL-time = xdf_mouse_mocap_time   ", xdf_mouse_mocap_time)
        print_stats("LSL-time = xdf_NIC_Quality_time   ", xdf_NIC_Quality_time)
        print_stats("BUG-time = xdf_kinect_mocap_time  ", xdf_kinect_mocap_time)
        print_stats("BUG-time = xdf_kinect_marker_time ", xdf_kinect_marker_time)
        print_stats("CSV-time = csv_kinect_mocap_time  ", csv_kinect_mocap_time)
        print("NOTE: Negative time values indicate missing streams")

    test_read_xdf_mouse_kinect()

### Get the kinect-to-csv delay


In [ ]:
def get_kinect_to_csv_delay(xdf_fullFname):
    """get the time difference between the xdf and csv mocap time"""

    xdf_mouse_kinect = read_xdf_mouse_kinect(xdf_fullFname)
    xdf_kinect_mocap_time = xdf_mouse_kinect["xdf_kinect_mocap_time"]
    csv_kinect_mocap_time = xdf_mouse_kinect["csv_kinect_mocap_time"]

    # NOTE: sometimes, the Kinect spits only zeros for some frames... we need to remove them
    xdf_kinect_mocap_data = xdf_mouse_kinect["xdf_kinect_mocap_data"]
    # get the index of row in xdf_kinect_mocap_data that are filled with zeros
    zero_rows = np.all(xdf_kinect_mocap_data == 0, axis=1)
    zero_rows_indices = np.where(zero_rows)[0]

    if np.any(zero_rows):
        logging.warning(
            f"Found {np.sum(zero_rows)} rows filled with zeros in the xdf_kinect_mocap_data in '{xdf_fullFname}' "
        )

    # remove the corresponding rows from the xdf_kinect_mocap_time and csv_kinect_mocap_time
    xdf_kinect_mocap_time = np.delete(xdf_kinect_mocap_time, zero_rows_indices)
    csv_kinect_mocap_time = np.delete(csv_kinect_mocap_time, zero_rows_indices)

    kinect_to_csv_delay = csv_kinect_mocap_time - xdf_kinect_mocap_time
    return kinect_to_csv_delay


if doRunTests:

    def test_get_kinect_to_csv_delay():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_c.xdf"

        kinect_to_csv_delay = get_kinect_to_csv_delay(xdf_fullFname)

        print(f"kinect_to_csv_delay (mean): {np.mean(kinect_to_csv_delay):.3f} s")
        print(f"kinect_to_csv_delay  (std): {np.std(kinect_to_csv_delay):.6f} s")

        # plot the time difference
        plt.figure()
        t = np.arange(len(kinect_to_csv_delay))
        x = kinect_to_csv_delay - np.mean(kinect_to_csv_delay)
        plt.plot(t, x, ".", markersize=1)
        plt.title("Kinect to CSV time difference (mean centered)")
        plt.xlabel("Frame number")
        plt.ylabel("Time difference (s)")
        plt.grid()
        plt.show()

    test_get_kinect_to_csv_delay()

## Mouse to csv delay

The task is not as trivial as computing the kinect-to-csv delay, as we need to find the corresponding csv file(s) for the mouse marker stream.


### Read one markers csv file

In [ ]:
def read_marker_csv_file(full_fname_marker_csv):
    """Read one marker csv file and return a list of [timestamp, marker] pairs

    Parameters
    ----------
    full_fname_marker_csv : str
        Full filename of the marker csv file

    Returns
    -------

    timestamp_marker_list : list
        List of [timestamp, marker] pairs
            timestamp : float (in seconds)
            marker : str
    """

    if not full_fname_marker_csv.endswith(".csv"):
        raise ValueError("The file must be a csv file")

    if not is_lsl_mouse_marker_csv_file(full_fname_marker_csv):
        msg = f"{full_fname_marker_csv} is not a LSL Mouse marker csv file."
        raise ValueError(msg)

    # NOTE: some mouse markers lack the quotes around the multiline markers
    # we need to add the quotes around the multiline markers before loading the file
    with open(full_fname_marker_csv, "r") as fname:
        txt = fname.readlines()

    if len(txt) < 3:
        raise ValueError(
            f"The file {full_fname_marker_csv} is too short ({len(txt)} lines)"
        )

    lines = [line.split(",") for line in txt]

    # find the lines where token 3 is "\n" = start of a multiline marker
    for i in range(len(lines)):
        line = lines[i]
        # find the start of a multiline marker
        if len(line) == 3 and line[2] == "\n":
            # add a " before the end of the line
            txt[i] = txt[i][:-1] + '"\n'
            # find the end of the multiline marker
            for j in range(i + 1, len(lines)):
                # if we have a normal one-line-marker
                if len(lines[j]) == 3:
                    # add a " before the end of the line (of the previous line)
                    txt[j - 1] = txt[j - 1][:-1] + '"\n'
                    break
                # if are at the end of the file
                if j == len(lines) - 1 and len(lines[j]) != 3:
                    # add a " before the end of the line
                    txt[j] = txt[j][:-1] + '"\n'
                    break

    # write the modified file to a temporary file and load it with np.loadtxt
    with tempfile.NamedTemporaryFile(mode="w", delete=False) as fname:
        fname.writelines(txt)
        tempFileName = fname.name

    lines = np.loadtxt(
        fname=tempFileName, skiprows=3, delimiter=",", quotechar='"', dtype=str
    )

    # remove the first column (we shall need only the timestamp and the marker to compare with the xdf file)
    lines = np.delete(lines, 0, 1)

    time_stamps = lines[:, 0].astype(float)
    time_stamps = time_stamps / 1000.0

    # markers should be an array of lists (to mimic the xdf format)
    markers = []
    for line in lines:
        mk = str(line[1:])
        markers.append(mk)

    timestamp_marker_list = list(zip(time_stamps, markers))

    return timestamp_marker_list


def plot_markers_csv_time_difference(marker_list, title_txt=""):
    """plot the time difference between the markers in the list"""
    plt.figure()
    t = np.arange(len(marker_list))
    x = [data[0] for data in marker_list]
    dx = np.diff(x)
    dx = np.insert(dx, 0, np.nan)

    plt.plot(t, dx, ".", markersize=10)
    plt.xlabel("Marker")
    plt.ylabel("Time difference (s)")
    plt.grid()
    # set the xticks to the marker names
    xticks_labels = [data[1] for data in marker_list]
    plt.xticks(t, xticks_labels, rotation=90, ha="center", fontsize=8)
    # leave some room in the bottom for the xticks_labels
    plt.subplots_adjust(bottom=0.5)
    # set the title
    plt_title = "full-screen window to view Markers labels"
    if title_txt:
        plt_title = f"{title_txt}\n{plt_title}"
    plt.title(plt_title)

    plt.show()

    return dx


def print_markers_csv_time_difference(marker_list, dx, title_txt=""):
    """print the time difference between the markers in the list"""
    print(f"{title_txt}: time difference {len(marker_list)} markers")
    print("  DeltaT     Time        Marker")
    for i in range(len(marker_list)):
        print(f"{dx[i]:8.3f} {marker_list[i][0]:.3f} {marker_list[i][1]}")


if doRunTests:

    def test_readMarkerCsv():
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_mau_np.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_mau_p.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_sau_np.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_sau_p.csv"

        # csv_fullFname = '../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_l_m_np_c.csv' # empty file

        fpath, fname = os.path.split(csv_fullFname)

        try:
            csv_marker_data = read_marker_csv_file(csv_fullFname)
            delta_time = plot_markers_csv_time_difference(
                csv_marker_data,
                title_txt=f"{csv_fullFname}",
            )
            print_markers_csv_time_difference(
                csv_marker_data, delta_time, title_txt=f"{csv_fullFname}"
            )
        except Exception as e:
            raise e

    test_readMarkerCsv()

### Read all markers csv files in the folder

In [ ]:
def read_all_marker_csv_files(xdf_full_path):
    """Read all marker csv files in the visit path and return a list of [timestamp, marker]
    pairs

    Parameters
    ----------
    xdf_full_path : str
        Full path of the directory where the xdf file is located

    Returns
    -------
    all_timestamp_marker_list : list
        List of [timestamp, marker] pairs
            timestamp : float (in seconds)
            marker : str
    """

    # get the list of all marker csv files in the visit path
    marker_files = [
        os.path.join(xdf_full_path, f)
        for f in os.listdir(xdf_full_path)
        if f.endswith(".csv") and is_lsl_mouse_csv_file(os.path.join(xdf_full_path, f))
    ]

    # if the list is empty, return an empty list
    if not marker_files:
        msg = f"No marker csv files with '_l_m_' in the name were found in '{xdf_full_path}'"
        logging.warning(msg)
        print(msg)
        return []

    # read all the marker csv files
    marker_data_dict = {}
    for i in range(len(marker_files)):
        try:
            mouse_markers = read_marker_csv_file(marker_files[i])
            marker = {
                "data": mouse_markers,
                "start": mouse_markers[0][0],
                "path": marker_files[i],
            }
            marker_data_dict[i] = marker
        except Exception as e:
            msg = f"Error reading '{marker_files[i]}': {e}"
            logging.warning(msg)
            print(msg)

    # sort the markers by their start time
    sorted_marker_data = sorted(marker_data_dict.items(), key=lambda x: x[1]["start"])

    # make a single list of markers from the multiple csv files for this xdf file
    all_timestamp_marker_list = []
    for i in range(len(sorted_marker_data)):
        all_timestamp_marker_list.extend(sorted_marker_data[i][1]["data"])

    return all_timestamp_marker_list


if doRunTests:

    def test_readAllMarkerCsvs():
        xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/"
        xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle"

        xdf_path = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular"

        xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle"  # empty file ReArm_C1P02_20210715_V3_l_m_np_c.csv
        xdf_path = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching"

        xdf_path = "../dat/ReArm.lnk/DATA_Named/C1P31/V3/Reaching"

        xdf_dir = xdf_path.split("/")[-1]  # get the last part of the path

        all_marker_data = read_all_marker_csv_files(xdf_path)
        if all_marker_data:
            plot_markers_csv_time_difference(
                all_marker_data, f"{xdf_dir}/ — Mouse Markers from all csv files"
            )

    test_readAllMarkerCsvs()

### Read the mouse and kinect streams from one .xdf file

In [ ]:
def read_xdf_mouse_markers(xdf_full_fname):
    """Read the mouse marker stream in an xdf file and return a list of [timestamp, marker]
    pairs

    Parameters
    ----------
    xdf_full_fname : str
        Full filename of the xdf file

    Returns
    -------
    timestamp_marker_list : list
        List of [timestamp, marker] pairs
            timestamp : float (in seconds)
            marker : str
    """

    xdf_data, header = pyxdf.load_xdf(
        filename=xdf_full_fname,
        select_streams=[
            {"type": "Markers", "name": "Mouse"},
            {"type": "Markers", "name": "MouseMarkers"},  # new naming possible
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
        verbose=False,
    )

    mouse_markers = [
        xdf_stream
        for xdf_stream in xdf_data
        if xdf_stream["info"]["name"][0] == "Mouse"
        or xdf_stream["info"]["name"][0] == "MouseMarkers"
    ][0]

    xdf_mouse_marker_time = mouse_markers["time_stamps"]
    xdf_mouse_marker_data = mouse_markers["time_series"]

    # return a list of [timestamp, marker] pairs
    timestamp_marker_list = []
    for i in range(len(xdf_mouse_marker_time)):
        timestamp_marker_list.append(
            [xdf_mouse_marker_time[i], xdf_mouse_marker_data[i]]
        )

    return timestamp_marker_list


if doRunTests:

    def test_xdf_mouse_markers():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_c.xdf"

        fname = xdf_fullFname.split("/")[-1]
        title_txt = f"{fname} — Mouse Markers"

        xdf_mouse_maker_list = read_xdf_mouse_markers(xdf_fullFname)
        plot_markers_csv_time_difference(xdf_mouse_maker_list, title_txt)

    test_xdf_mouse_markers()

### [Functions to compare [timestamp, marker ] lists](#toc0_)

The human-readable markers streams are stored in lists of [timestamp, marker].   
We need to compare the list from the xdf and the list from the csv file(s) to ensure that we map the correct markers to the correct timestamps in both lists.


In [ ]:
def define_shortest_longest_lists(list1, list2):
    """Define the shortest and the longest list"""

    shortest_list = list1
    longest_list = list2
    if len(list2) < len(list1):
        shortest_list = list2
        longest_list = list1

    return shortest_list, longest_list


def index_of_shortest_in_longest_lists(shortest_list, longest_list):
    """Find the start and stop of the shortest list (as a block of lines) in the longest list using difflib"""

    # NOTE: we use difflib because:
    # - it can find contiguous common blocks of lines
    # - it is easy to visually check the results with the diff output

    # NOTE: we expect the shortest list to be a single block of lines in the longest list

    # get two temporary files to write the lists of markers for difflib
    with tempfile.NamedTemporaryFile(mode="w", delete=False) as short_file:
        for item in shortest_list:
            short_file.write(f"{item[1]}\n")
        short_file_name = short_file.name

    with tempfile.NamedTemporaryFile(mode="w", delete=False) as long_file:
        for item in longest_list:
            long_file.write(f"{item[1]}\n")
        long_file_name = long_file.name

    # read the two temporary files
    with open(short_file_name, "r") as f:
        shortest = f.readlines()

    with open(long_file_name, "r") as f:
        longest = f.readlines()

    # Use difflib to compare the two lists of lines
    d = difflib.Differ()
    diff = d.compare(shortest, longest)

    # Collect contiguous common blocks with line numbers
    common_blocks = []
    block_lines = []  # To track line numbers
    block = []

    for i, line in enumerate(diff):
        # Lines that are the same in both files
        if line.startswith(" "):
            block.append(line[2:])  # Skip the leading ' ' in the diff output
            block_lines.append(i)  # Line number in longest
        # When we encounter a non-common line and we have accumulated a block
        elif block:
            common_blocks.append((block, block_lines))
            # Reset the block
            block = []
            block_lines = []

    # If there's a block at the end, add it
    if block:
        common_blocks.append((block, block_lines))

    # Remove the temporary files
    os.remove(short_file_name)
    os.remove(long_file_name)

    # check if we have found the shortest list in the longest list
    if len(common_blocks) == 0:
        raise ValueError("No common blocks found between the two lists")

    if len(common_blocks) != 1:
        raise ValueError("The shortest list is not a single block in the longest list")

    # we have a single block of lines
    common_block = common_blocks[0]

    common_block_lines = common_block[1]
    i_beg = common_block_lines[0]
    i_end = common_block_lines[-1]

    return i_beg, i_end


def get_timestamps_differences(longest_list, shortest_list, i_beg, i_end):
    """Get the differences between the timestamps of the common part and the shortest list"""

    if len(shortest_list) > len(longest_list):
        raise ValueError(
            "The shortest list (second argument) must be shorter than the longest list"
        )

    timestamps_common_part = [x[0] for x in longest_list[i_beg : i_end + 1]]
    timestamps_shortest_list = [x[0] for x in shortest_list]

    timestamps_differences = [
        timestamps_common_part[i] - timestamps_shortest_list[i]
        for i in range(len(shortest_list))
    ]

    # NOTE: we do not know whether csv or xdf is in the shortest list, hence we do not know the sign!
    # Too bad... BUT...
    # timestamps_differences MUST be positive values.
    # This is because csv time is UNIX time (seconds since 1970) and
    # xdf time is in seconds since the start of the recording (or something like that).

    timestamps_differences = [abs(x) for x in timestamps_differences]

    return timestamps_differences


def index_of_longest_block_common_to_shortest_and_longest_lists(
    shortest_list_, longest_list_
):

    # NOTE: Here we want more than the markers: we can also include the timestamps differences
    # to the marker so that ['KeyTyped=113 WINDOW_CLOSING'] becomes [5.94, 'KeyTyped=113 WINDOW_CLOSING']
    # To do so:
    # we replace the timestamps by the differences between the timestamps, rounded to the nearest 1/100s
    # this is because the timestamps in the xdf files are not the same as the timestamps in the csv files
    # but the differences between the timestamps are the same in both files, rounded to the nearest 1/100s

    # make a copy of the lists (we shall modify them)
    shortest_list = shortest_list_.copy()
    longest_list = longest_list_.copy()

    # compute the time difference between the timestamps of the two lists
    short_timestamps = [x[0] for x in shortest_list]
    long_timestamps = [x[0] for x in longest_list]

    short_diff_timestamps = np.diff(short_timestamps)
    long_diff_timestamps = np.diff(long_timestamps)
    # append a zero at the beginning of the differences
    short_diff_timestamps = np.insert(short_diff_timestamps, 0, 0)
    long_diff_timestamps = np.insert(long_diff_timestamps, 0, 0)

    # round the differences to the nearest 1/100s or so
    # this is to account for possible transmission delays
    # NOTE: the allowed error has almost no effect on kinect time correction (5th decimal)
    allowed_error = 0.1  # common blocks are longer with this value
    short_diff_timestamps = allowed_error * np.round(
        short_diff_timestamps / allowed_error
    )
    long_diff_timestamps = allowed_error * np.round(
        long_diff_timestamps / allowed_error
    )

    # replace the timestamps by the differences
    shortest_list = [
        [short_diff_timestamps[i], shortest_list[i][1]]
        for i in range(len(shortest_list))
    ]
    longest_list = [
        [long_diff_timestamps[i], longest_list[i][1]] for i in range(len(longest_list))
    ]

    # TODO: clean the debug code
    # print the lists in a file for visual inspection
    with open("../debug/shortest_list.csv", "w") as f:
        for item in shortest_list:
            f.write(f"{item[0]},{item[1]}\n")

    with open("../debug/longest_list.csv", "w") as f:
        for item in longest_list:
            f.write(f"{item[0]},{item[1]}\n")

    # NOTE: we use difflib because:
    # - it can find contiguous common blocks of lines
    # - it is easy to visually check the results with the diff output

    # NOTE: we expect the shortest list to be a single block of lines in the longest list
    # Yet, this is not always the case...
    # hence we find the longest block that is common to the longest list and the shortest list

    # get two temporary files to write the lists of markers
    with tempfile.NamedTemporaryFile(mode="w", delete=False) as short_file:
        for item in shortest_list:
            short_file.write(f"{item[0]}{item[1]}\n")
        short_file_name = short_file.name

    with tempfile.NamedTemporaryFile(mode="w", delete=False) as long_file:
        for item in longest_list:
            long_file.write(f"{item[0]}{item[1]}\n")
        long_file_name = long_file.name

    # read the two temporary files
    with open(short_file_name, "r") as f:
        shortest = f.readlines()

    with open(long_file_name, "r") as f:
        longest = f.readlines()

    # Use difflib to compare the two lists of lines
    d = difflib.Differ()
    diff = d.compare(shortest, longest)

    # print the diff output in a file for visual inspection
    diff_ = d.compare(shortest, longest)  # copy the iterator
    with open("../debug/diff_output.txt", "w") as f:
        f.writelines(diff_)

    # Collect contiguous common blocks with line numbers
    common_blocks = []
    block = {
        "lines": [],
        "i_common_block_in_longest": [],
        "i_common_block_in_shortest": [],
    }

    # diff output format:
    # ' ' : line common to both files
    # '-' : line only in the first file (shortest)
    # '+' : line only in the second file (longest)

    i_in_shortest = 0
    i_in_longest = 0
    for i, line in enumerate(diff):
        # Lines that are the same in both files : common block
        if line[0] == " ":
            common_text = line[2:]
            # add the line and increase the counter in both list
            block["lines"].append(common_text)
            block["i_common_block_in_longest"].append(i_in_longest)
            block["i_common_block_in_shortest"].append(i_in_shortest)
            i_in_shortest += 1
            i_in_longest += 1
        # else, we have a new block : save the previous block and start a new one
        else:
            # if the block is not empty, save it and start a new one
            if block["lines"]:
                common_blocks.append(block)
                block = {
                    "lines": [],
                    "i_common_block_in_longest": [],
                    "i_common_block_in_shortest": [],
                }
            if line[0] == "+":
                i_in_longest += 1
            if line[0] == "-":
                i_in_shortest += 1
        # if all lines ane common, we need to save the last block
        if block["lines"]:
            common_blocks.append(block)

    if len(shortest) == 0:
        raise ValueError("The shortest list is empty")

    if len(common_blocks) == 0:
        raise ValueError("No common blocks found between the two lists")

    # get the longest block in common_blocks
    longest_block = max(common_blocks, key=lambda x: len(x["lines"]))

    # Remove the temporary files
    os.remove(short_file_name)
    os.remove(long_file_name)

    indexes = {
        "i_beg_in_long_list": longest_block["i_common_block_in_longest"][0],
        "i_end_in_long_list": longest_block["i_common_block_in_longest"][-1],
        "i_beg_in_short_list": longest_block["i_common_block_in_shortest"][0],
        "i_end_in_short_list": longest_block["i_common_block_in_shortest"][-1],
    }

    return indexes


def get_timestamps_differences_2(longest_list, shortest_list, indexes):
    """Get the differences between the timestamps of the common part and the shortest list"""

    if len(shortest_list) > len(longest_list):
        raise ValueError(
            "The shortest list (second argument) must be shorter than the longest list"
        )

    timestamps_common_part = [
        longest_list[i][0]
        for i in range(indexes["i_beg_in_long_list"], indexes["i_end_in_long_list"] + 1)
    ]
    timestamps_shortest_list = [
        shortest_list[i][0]
        for i in range(
            indexes["i_beg_in_short_list"], indexes["i_end_in_short_list"] + 1
        )
    ]

    if len(timestamps_common_part) != len(timestamps_shortest_list):
        raise ValueError(
            "The common part of the two lists must have the same length as the shortest list"
        )

    timestamps_differences = [
        timestamps_common_part[i] - timestamps_shortest_list[i]
        for i in range(len(timestamps_common_part))
    ]

    # NOTE: we do not know whether csv or xdf is in the shortest list, hence we do not know the sign!
    # Too bad... BUT...
    # timestamps_differences MUST be positive values.
    # This is because csv time is UNIX time (seconds since 1970) and
    # xdf time is in seconds since the start of the recording (or something like that).

    timestamps_differences = [abs(x) for x in timestamps_differences]

    return timestamps_differences

### Get the mouse-to-csv delay

In [ ]:
def get_mouse_to_csv_delay_list(xdf_mouse_marker_list, csv_mouse_marker_list):
    """Get the mouse-to-csv delay as a list of [timestamp, marker] pairs"""

    # find the shortest and the longest list
    shortest_list, longest_list = define_shortest_longest_lists(
        xdf_mouse_marker_list, csv_mouse_marker_list
    )

    indexes = index_of_longest_block_common_to_shortest_and_longest_lists(
        shortest_list, longest_list
    )

    # get the differences between the timestamps of the common part and the shortest list
    timestamps_differences = get_timestamps_differences_2(
        longest_list, shortest_list, indexes
    )
    # as np array
    timestamps_differences = np.array(timestamps_differences)

    # get the makers corresponding to the timestamps_differences
    markers_differences = [
        shortest_list[i][1]
        for i in range(
            indexes["i_beg_in_short_list"], indexes["i_end_in_short_list"] + 1
        )
    ]
    # make it a list of [timestamp, marker] pairs
    timestamps_differences = list(zip(timestamps_differences, markers_differences))

    return timestamps_differences


if doRunTests:

    def test_get_mouse_to_csv_delay():

        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        # xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_c.xdf"

        # xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf"
        xdf_path = os.path.dirname(xdf_fullFname)

        # get the csv marker data
        csv_mouse_marker_list = read_all_marker_csv_files(xdf_path)
        if not csv_mouse_marker_list:
            return
        csv_mouse_marker_time = [data[0] for data in csv_mouse_marker_list]
        csv_mouse_marker_data = [data[1] for data in csv_mouse_marker_list]

        xdf_mouse_kinect = read_xdf_mouse_kinect(xdf_fullFname)
        xdf_mouse_marker_time = xdf_mouse_kinect["xdf_mouse_marker_time"]
        xdf_mouse_marker_data = xdf_mouse_kinect["xdf_mouse_marker_data"]

        xdf_mouse_marker_list = list(zip(xdf_mouse_marker_time, xdf_mouse_marker_data))

        # get the mouse-to-csv delay as a list of [timestamp, marker] pairs
        mouse_to_csv_delay_list = get_mouse_to_csv_delay_list(
            xdf_mouse_marker_list, csv_mouse_marker_list
        )

        # get the mouse-to-csv delay timestamps
        mouse_to_csv_delay = [data[0] for data in mouse_to_csv_delay_list]
        # as a np array
        mouse_to_csv_delay = np.array(mouse_to_csv_delay)

        print(f"mouse_to_csv_delay (median): {np.median(mouse_to_csv_delay):.3f} s")
        print(f"mouse_to_csv_delay   (mean): {np.mean(mouse_to_csv_delay):.3f} s")
        print(f"mouse_to_csv_delay    (std): {np.std(mouse_to_csv_delay):.6f} s")

        # plot the time difference
        t = np.arange(len(mouse_to_csv_delay))
        x = mouse_to_csv_delay - np.mean(mouse_to_csv_delay)

        fig, ax = plt.subplots(1, 2, width_ratios=[10, 1], figsize=(12, 6))
        # scatter plot
        ax[0].plot(t, x, ".", markersize=10)
        ax[0].set_title(
            f"Mouse to CSV time difference mean = {np.mean(mouse_to_csv_delay):.3f} s"
        )
        ax[0].set_xlabel("Marker number")
        ax[0].set_ylabel("Time difference (mean centered, s)")
        ax[0].grid()

        # leave some room in the bottom for the xticks_labels
        plt.subplots_adjust(bottom=0.5)
        # add the xticks_labels
        xticks_labels = [data[1][0] for data in mouse_to_csv_delay_list]
        ax[0].set_xticks(t)
        ax[0].set_xticklabels(xticks_labels, rotation=90, ha="center", fontsize=8)

        # boxplot
        ax[1].boxplot(x, showmeans=True)
        # # remove the frame around the boxplot
        ax[1].axis("off")
        # ensure the x limits are the same for both plots
        ax[1].set_ylim(ax[0].get_ylim())

        plt.show()

    test_get_mouse_to_csv_delay()

## Kinect-to-mouse delay

In [ ]:
# get the kinect-to-mouse delay
#   kinect_to_mouse_delay_s = kinect_to_csv_delay_s - mouse_to_csv_delay_s


def get_kinect_to_mouse_delay(kinect_to_csv_delay, mouse_to_csv_delay):
    """Get the kinect-to-mouse delay"""

    kinect_to_cvs_delay_median = np.median(kinect_to_csv_delay)
    kinect_to_cvs_delay_mean = np.mean(kinect_to_csv_delay)
    kinect_to_cvs_delay_std = np.std(kinect_to_csv_delay)

    mouse_to_csv_delay_median = np.median(mouse_to_csv_delay)
    mouse_to_csv_delay_mean = np.mean(mouse_to_csv_delay)
    mouse_to_csv_delay_std = np.std(mouse_to_csv_delay)

    # NOTE: the kinect to mouse delay is the difference of the means-medians
    # The mean & std is the simplest approach, if the distribution is normal
    # which should be true, as the error is due to random delays in the system
    kinect_to_mouse_delay_mean = kinect_to_cvs_delay_mean - mouse_to_csv_delay_mean

    kinect_to_mouse_delay_median = (
        kinect_to_cvs_delay_median - mouse_to_csv_delay_median
    )

    # NOTE: the variance of the difference is the sum of the variances **minus the covariance**
    # (e.g. https://en.wikipedia.org/wiki/Propagation_of_uncertainty)
    # If we assume that the two distributions are independent, the covariance is zero
    # hence computing the variance of the difference as the sum of the variances is correct.
    # If we assume that the two distributions are not independent, we should subtract the covariance,
    # but we do not have it. We still can compute the variance of the difference as the sum of the variances,
    # and this will be an **upper bound of the variance of the difference**.
    kinect_to_mouse_delay_std = np.sqrt(
        kinect_to_cvs_delay_std**2 + mouse_to_csv_delay_std**2
    )

    return (
        kinect_to_mouse_delay_mean,
        kinect_to_mouse_delay_median,
        kinect_to_mouse_delay_std,
    )


def plot_delay_distribution(
    kinect_to_csv_delay,
    mouse_to_csv_delay,
):
    """Boxplot the distribution of the kinect-to-csv and mouse-to-csv delays"""

    # boxplot the two distributions (mean centered)
    k_distrib = kinect_to_csv_delay - np.mean(kinect_to_csv_delay)
    m_distrib = mouse_to_csv_delay - np.mean(mouse_to_csv_delay)

    plt.figure()
    plt.boxplot([k_distrib, m_distrib], showmeans=True)
    plt.title("Kinect to CSV delay vs Mouse to CSV delay")
    plt.xticks([1, 2], ["Kinect to CSV", "Mouse to CSV"])
    plt.ylabel("Time difference, mean centered (s)")
    plt.grid()
    # add the 95% confidence interval

    kinect_to_cvs_delay_std = np.std(kinect_to_csv_delay)
    mouse_to_csv_delay_std = np.std(mouse_to_csv_delay)

    plt.errorbar(
        [1.2, 2.2],
        [0, 0],
        yerr=[1.96 * kinect_to_cvs_delay_std, 1.96 * mouse_to_csv_delay_std],
        fmt="o",
        color="b",
        label="95% confidence interval",
    )
    plt.legend()
    plt.show()


def get_kinect_to_mouse_delay_for_xdf(xdf_fullFname):
    """Get the kinect-to-mouse delay for the xdf file in a dict containing :
    - "kinect_to_mouse_delay_mean": float
    - "kinect_to_mouse_delay_median": float
    - "kinect_to_mouse_delay_std": float
    - "kinect_to_csv_delay": np.array
    - "mouse_to_csv_delay": np.array
    """
    xdf_path = os.path.dirname(xdf_fullFname)

    # get the csv marker data
    csv_mouse_marker_list = read_all_marker_csv_files(xdf_path)
    if not csv_mouse_marker_list:

        return {
            "kinect_to_mouse_delay_mean": np.nan,
            "kinect_to_mouse_delay_median": np.nan,
            "kinect_to_mouse_delay_std": np.nan,
            "kinect_to_csv_delay": np.nan,
            "mouse_to_csv_delay": np.nan,
        }

    xdf_mouse_kinect = read_xdf_mouse_kinect(xdf_fullFname)
    xdf_mouse_marker_time = xdf_mouse_kinect["xdf_mouse_marker_time"]
    xdf_mouse_marker_data = xdf_mouse_kinect["xdf_mouse_marker_data"]

    xdf_mouse_marker_list = list(zip(xdf_mouse_marker_time, xdf_mouse_marker_data))

    # get the mouse-to-csv delay
    mouse_to_csv_delay_list = get_mouse_to_csv_delay_list(
        xdf_mouse_marker_list, csv_mouse_marker_list
    )
    mouse_to_csv_delay = np.array([data[0] for data in mouse_to_csv_delay_list])

    # get the kinect-to-csv delay
    kinect_to_csv_delay = get_kinect_to_csv_delay(xdf_fullFname)

    # get the kinect-to-mouse delay
    (
        kinect_to_mouse_delay_mean,
        kinect_to_mouse_delay_median,
        kinect_to_mouse_delay_std,
    ) = get_kinect_to_mouse_delay(kinect_to_csv_delay, mouse_to_csv_delay)

    return {
        "kinect_to_mouse_delay_mean": kinect_to_mouse_delay_mean,
        "kinect_to_mouse_delay_median": kinect_to_mouse_delay_median,
        "kinect_to_mouse_delay_std": kinect_to_mouse_delay_std,
        "kinect_to_csv_delay": kinect_to_csv_delay,
        "mouse_to_csv_delay": mouse_to_csv_delay,
    }


if doRunTests:

    def test_get_kinect_to_mouse_time_delay():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Circle/ReArm_C1P02_20210409_V2_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Reaching/ReArm_C1P02_20210409_V2_r.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V3/Reaching/task-V3_Reach.xdf"  # new format

        delays_for_xdf = get_kinect_to_mouse_delay_for_xdf(xdf_fullFname)
        kinect_to_mouse_delay_mean = delays_for_xdf["kinect_to_mouse_delay_mean"]
        kinect_to_mouse_delay_median = delays_for_xdf["kinect_to_mouse_delay_median"]
        kinect_to_mouse_delay_std = delays_for_xdf["kinect_to_mouse_delay_std"]
        kinect_to_csv_delay = delays_for_xdf["kinect_to_csv_delay"]
        mouse_to_csv_delay = delays_for_xdf["mouse_to_csv_delay"]

        print(f"kinect_to_mouse_delay (median): {kinect_to_mouse_delay_median:.3f} s")
        print(f"kinect_to_mouse_delay   (mean): {kinect_to_mouse_delay_mean:.3f} s")
        print(f"kinect_to_mouse_delay    (std): {kinect_to_mouse_delay_std:.6f} s")

        # display the kinect to mouse delay
        plot_delay_distribution(
            kinect_to_csv_delay,
            mouse_to_csv_delay,
        )

    test_get_kinect_to_mouse_time_delay()

# Test the solution on one file



## Utility functions for .xdf files

In [ ]:
def print_streams_types_and_names(fullFname_or_streams):
    """Print the names and types of all streams in the xdf file or in the streams list"""

    if isinstance(fullFname_or_streams, str):
        xdf_data, header = pyxdf.load_xdf(filename=fullFname_or_streams, verbose=False)
    elif (
        isinstance(fullFname_or_streams, list)
        and all(isinstance(x, dict) for x in fullFname_or_streams)
        and all("info" in x for x in fullFname_or_streams)
    ):
        xdf_data = fullFname_or_streams
    else:
        raise ValueError("The first argument must be a filename or a list of streams")

    for i in range(len(xdf_data)):
        stream = xdf_data[i]
        s_type = stream["info"]["type"][0]
        s_name = stream["info"]["name"][0]
        print(f"Stream {i}: {s_type}, {s_name}")


def get_stream(xdf_data, searched_stream_type, searched_stream_names):
    """Get the stream of type 'searched_stream_type' with name in 'searched_stream_names' in the xdf_data"""

    if not isinstance(
        searched_stream_names, list
    ):  # if we get a string (only one name)
        searched_stream_names = [searched_stream_names]

    found_streams = []
    for stream in xdf_data:
        stream_type = stream["info"]["type"][0]
        if searched_stream_type == stream_type:
            stream_name = stream["info"]["name"][0]
            for searched_stream_name in searched_stream_names:
                if searched_stream_name == stream_name:
                    found_streams.append(stream)

    if not found_streams:
        msg = f" Stream not found. Searched in [{searched_stream_type}:{searched_stream_names}]."
        logging.warning(msg)
        print(msg)
        return None

    if len(found_streams) > 1:
        found_streams_names = [stream["info"]["name"][0] for stream in found_streams]
        msg = f"Found multiple streams: [{searched_stream_type},{found_streams_names}]."
        raise ValueError(msg)

    return found_streams[0]


def get_kinect_channel(kinect_mocap, channel_name):
    """Get one channel from the kinect mocap by its name"""
    channel_index = -1
    nb_channels = len(kinect_mocap["info"]["desc"][0]["channels"][0]["channel"])
    for i in range(nb_channels):
        current_name = kinect_mocap["info"]["desc"][0]["channels"][0]["channel"][i][
            "label"
        ][0]
        if current_name == channel_name:
            channel_index = i
            break
    if channel_index == -1:
        raise ValueError(f"Joint {channel_name} not found in the kinect mocap data")

    channel_data = kinect_mocap["time_series"][:, channel_index]

    return channel_data


if doRunTests:

    def test_get_stream():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"

        xdf_data, header = pyxdf.load_xdf(filename=xdf_fullFname, verbose=False)

        # print_streams_names_type(xdf_fullFname)
        print_streams_types_and_names(xdf_data)

        searched_type = "Markers"
        # stream_names_list = ["Mouse", "MouseMarkers"]  # raise ValueError ['Mouse', 'MouseToNIC']
        # searched_names = ["toto", "titi"]  # raise ValueError
        searched_names = ["Mouse", "Mouse-Markers"]  # OK : [Markers, Mouse]

        stream = get_stream(xdf_data, searched_type, searched_names)
        if stream:
            found_stream_name = stream["info"]["name"][0]
            found_stream_type = stream["info"]["type"][0]
            print(f"Found stream: [{found_stream_type}, {found_stream_name}]")
        else:
            print("Stream not found")

    test_get_stream()

## Check if the xdf file has a kinect stream with bugged timestamps


 There is a bug if the range of the kinect timestamps differs from the range of the other LSL timestamps.

 We use the NIC Quality stream as a reference, as it is always present in the ReArm data set.

 Typical differences are: 

 * If there is a bug: 
    ```
    LSL-time = xdf_NIC_Quality_time   :       22886.697s to       23270.697s,    385 samples for a duration of  384.000s 
    BUG-time = xdf_kinect_mocap_time  :          39.798s to         416.469s,  11277 samples for a duration of  376.671s 
    ```

* If there is no bug: 
    ```
    LSL-time = xdf_NIC_Quality_time   :     3629041.763s to     3629294.763s,    254 samples for a duration of  253.000s
    BUG-time = xdf_kinect_mocap_time  :     3629041.174s to     3629295.670s,   5622 samples for a duration of  254.496s
    ``` 

We decide that there is no bug if the difference between the two ranges is less than 1 hour (3600s).

In [ ]:
def debug_needs_kinect_timestamps_correction(xdf_fullFame):

    # (re)load all streams
    streams, fileheader = pyxdf.load_xdf(
        filename=xdf_fullFame,
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps
        verbose=False,
    )
    # Print the start time of all streams
    for stream in streams:
        start_time = stream["time_stamps"][0]
        name = stream["info"]["name"][0]
        duration = stream["time_stamps"][-1] - start_time
        print(f"start at {start_time:15.3f} for {duration:7.3f} s : {name}")

    # boxplot of the timestamps by stream
    timestamps = [stream["time_stamps"] for stream in streams]

    plt.figure()
    plt.boxplot(timestamps, showmeans=True)
    plt.title("Timestamps of all streams")
    plt.xticks(
        range(1, len(streams) + 1), [stream["info"]["name"][0] for stream in streams]
    )
    plt.ylabel("Time (s)")
    plt.grid()
    plt.show()

    # boxplot of the timestamps for the streams whose name contains kinect vs. the other streams
    kinect_streams = [
        stream for stream in streams if "Kinect" in stream["info"]["name"][0]
    ]

    if not kinect_streams:
        print("No Kinect stream found")
        return

    kinect_timestamps = [stream["time_stamps"] for stream in kinect_streams]
    # merge the timestamps of the kinect streams
    kinect_timestamps = np.concatenate(kinect_timestamps)

    other_streams = [
        stream for stream in streams if "Kinect" not in stream["info"]["name"][0]
    ]
    other_timestamps = [stream["time_stamps"] for stream in other_streams]
    # merge the timestamps of the other streams
    other_timestamps = np.concatenate(other_timestamps)

    plt.figure()
    plt.boxplot([kinect_timestamps, other_timestamps], showmeans=True)
    plt.title("Timestamps of the Kinect and other streams")
    plt.xticks([1, 2], ["Kinect", "Other streams"])
    plt.ylabel("Time (s)")
    plt.grid()
    plt.show()


def needs_kinect_timestamps_correction(xdf_fullFame, do_debug_msg="do_not_debug"):
    """
    Check if the timestamps of the Kinect stream need to be corrected.
    The timestamps of the Kinect stream are compared to the timestamps of the NIC-Quality stream.
    The timestamps of the NIC-Quality stream are considered as the reference.

    returns:
    - True if the timestamps of the Kinect stream need to be corrected
    - False if the timestamps of the Kinect stream are in the range of the NIC-Quality timestamps
    """

    # NOTE: we must not use the dejitter_timestamps=True because :
    #  - the EuroMov-Mocap-Kinect stream is not a regular stream (either 30Hz or 15 Hz, depending on the light conditions)
    #  - the dejitter_timestamps=True will interpolate the timestamps of the Kinect stream (we do not want that)
    #  - we want to compare the raw timestamps of the Kinect stream with the raw timestamps of the NIC-Quality stream

    # NIC-Quality stream is always present in the xdf files (and it is fast to load)

    streams, fileheader = pyxdf.load_xdf(
        filename=xdf_fullFame,
        select_streams=[  # select only the streams we need = fast
            {"type": "Quality"},  # NIC changed the name: NIC or LSLOutletStreamName
            {"type": "MoCap"},  # Kinect (and Mouse)
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps
        verbose=False,
    )
    if doRunTests and do_debug_msg == "do_debug":
        debug_needs_kinect_timestamps_correction(xdf_fullFame)

    nic_quality_stream = [
        stream for stream in streams if stream["info"]["type"][0] == "Quality"
    ]

    kinect_mocap_stream = [
        stream
        for stream in streams
        if stream["info"]["name"][0] == "EuroMov-Mocap-Kinect"
    ]

    if not nic_quality_stream:
        print("Stream 'Quality' not found: ")
        print_streams_types_and_names(xdf_fullFame)
        raise ValueError("Stream 'Quality' not found in the xdf data")

    if not kinect_mocap_stream:
        msg = "    there is no need to correct a stream that is missing..."
        logging.info(msg)
        print(msg)
        return False

    # get the timestamps
    nic_quality_timestamps = nic_quality_stream[0]["time_stamps"]
    kinect_mocap_timestamps = kinect_mocap_stream[0]["time_stamps"]

    # check if the streams are not empty or quasi-empty
    if kinect_mocap_timestamps.size < 10:
        msg = "    there is no need to correct an empty stream..."
        logging.info(msg)
        print(msg)
        return False

    # NOTE: if the iqr of the kinect mocap timestamps and the iqr of the nic-quality timestamps do not overlap
    # then the kinect timestamps need to be corrected
    # --> we do not use kinect markers that are usually *only at the beginning*

    # get the iqr of the timestamps
    kinect_mocap_iqr = np.percentile(kinect_mocap_timestamps, [25, 75])
    nic_quality_iqr = np.percentile(nic_quality_timestamps, [25, 75])

    # check if the iqr overlap
    needs_correction = (
        kinect_mocap_iqr[1] < nic_quality_iqr[0]
        or kinect_mocap_iqr[0] > nic_quality_iqr[1]
    )

    return needs_correction


if doRunTests:

    def test_check_kinect_timestamps():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        # xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Reaching/ReArm_C1P42_20240603_V1_r.xdf" # Kinect markers with 1 value that is empty
        # xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf" # Kinect markers with 1 value that is empty
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_c.xdf"

        # xdf_fullFname = (
        #     "../dat/ReArm.lnk/DATA_named/C1P02/V2/Armeo/002_CorJea_20210409_2_a.xdf"
        # )
        # xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P07/V3/Reaching/C1P07_MagPau_20211116_3_r.xdf" #  correction needed

        # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P14/V2/Circular/C1P14_CoqNel_20220128_2_c.xdf"  #  correction needed
        # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P45/V1/Circular/task-V1_Circle.xdf"  # no correction needed

        xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V3/Reaching/task-V3_Reach.xdf"  # new format

        needs_correction = needs_kinect_timestamps_correction(xdf_fullFname, "do_debug")
        if needs_correction:
            print("Kinect timestamps need to be corrected")
        else:
            print(
                "Kinect timestamps are in the inter-quartile range of NIC-Quality timestamps"
            )

    test_check_kinect_timestamps()

## Test the solution on one .xdf circle file

The circle tasks records the hand movement of the participant drawing a circle. The same hand movement is recorded by the kinect and the mouse, hence allowing visual inspection of the corrected kinect timestamps.

In [ ]:
from scipy.signal import butter, filtfilt


def plot_start_stop_times(start_times, stop_times):
    """Plot the start and stop times as vertical lines"""
    # plot the start times as a vertical line
    for t_start in start_times:
        plt.axvline(x=t_start, color="red", linestyle="--")
    # plot the stop times as a vertical line
    for t_stop in stop_times:
        plt.axvline(x=t_stop, color="green", linestyle="--")


def plot_t_xyz(t, x, y, z, name, start_times=None, stop_times=None):
    """Plot the x, y, z data as a function of time", name is the name of the data"""
    plt.plot(t, x, ".", markersize=2, label=f"x.{name}")
    plt.plot(t, y, ".", markersize=2, label=f"y.{name}")
    plt.plot(t, z, ".", markersize=2, label=f"z.{name}")
    if start_times is not None and stop_times is not None:
        plot_start_stop_times(start_times, stop_times)
    plt.xlabel("Time (s)")
    plt.ylabel("Position (m)")
    plt.legend()
    plt.grid()


def plot_trajectory(t, x, y, z, t_start, t_stop, name):
    """Plot the 3D trajectory of the x, y, z data"""

    # restrict the zone of interest from t_start to t_stop
    i_start = np.where(t >= t_start)[0][0]
    i_stop = np.where(t <= t_stop)[0][-1]
    x = x[i_start:i_stop]
    y = y[i_start:i_stop]
    z = z[i_start:i_stop]

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(x, y, z, ".", markersize=2)
    ax.set_xlabel(f"x.{name}")
    ax.set_ylabel(f"y.{name}")
    ax.set_zlabel(f"z.{name}")
    plt.title(f"{name} 3D trajectory")
    plt.show()


def low_pass_butt(signal, cutoff, fs, order=2):
    """Low pass filter the signal using a Butterworth filter"""
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(N=order, Wn=normal_cutoff, btype="low", analog=False)
    y = filtfilt(b, a, signal)
    return y


def find__mouse_marker_indexes(marker_name, markers_data):
    """Find the indexes of the marker in the markers data"""

    marker_index_list = [
        i for i, marker in enumerate(markers_data) if marker_name in marker[0]
    ]

    return marker_index_list


def visually_check_time_correction(xdf_fullFname, option_str="show_plot"):
    """Visualy check the time correction of the Kinect data"""

    # read the xdf file
    xdf_data, header = pyxdf.load_xdf(
        filename=xdf_fullFname,
        synchronize_clocks=True,
        #
        select_streams=[
            {"type": "MoCap"},  # kinect and mouse
            {"type": "Accelerometer"},
            {"type": "Markers"},
        ],
        dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
        verbose=False,
    )

    mouse_mocap = get_stream(xdf_data, "MoCap", ["Mouse", "MouseData"])
    kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")

    # NIC changed the name to LSLOutletStreamName
    s_names = ["NIC-Accelerometer", "LSLOutletStreamName-Accelerometer"]
    acc_mocap = get_stream(xdf_data, "Accelerometer", s_names)

    nic_markers = get_stream(
        xdf_data, "Markers", ["NIC-Markers", "LSLOutletStreamName-Markers"]
    )

    mouse_markers = get_stream(xdf_data, "Markers", ["MouseMarkers", "Mouse"])
    # kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")

    mouse_to_nic_markers = get_stream(xdf_data, "Markers", "MouseToNIC")

    event_ide_TONIC = get_stream(xdf_data, "Markers", "event_ide_TONIC")
    condition_reaching_task = get_stream(xdf_data, "Markers", "condition_reaching_task")

    def print_markers_data(markers_data, name):
        """Print the markers data"""
        print(f"{name}")
        for i in range(len(markers_data["time_stamps"])):
            print(
                f"{markers_data['time_stamps'][i]:.3f} {markers_data['time_series'][i]}"
            )
        print()

    def print_and_plot_markers_stream(markers_stream, title_txt=""):
        """Print and plot the markers data"""
        time_stamps = markers_stream["time_stamps"]
        time_series = markers_stream["time_series"]
        stream_name = markers_stream["info"]["name"][0]
        n_markers = len(time_stamps)

        if title_txt == "":
            title_txt = f"{stream_name} ({n_markers} markers)"

        timestamp_marker_list = list(zip(time_stamps, time_series))
        print_markers_data(markers_stream, title_txt)
        plot_markers_csv_time_difference(timestamp_marker_list, title_txt=title_txt)

    def plot_two_marker_lists(markers_stream1, markers_stream2, title_txt=""):
        """Plot the time difference between two marker lists"""

        stream_name1 = markers_stream1["info"]["name"][0]
        time_stamps1 = markers_stream1["time_stamps"]
        time_series1 = markers_stream1["time_series"]
        n_markers1 = len(time_stamps1)

        # ensure the time series contains strings
        if not isinstance(time_series1[0], str):
            time_series1 = [str(x) for x in time_series1]

        # ensure the time series is a numpy array
        if isinstance(time_series1, list):
            time_series1 = np.array(time_series1)

        time_series1 = stream_name1 + " " + np.array(time_series1)

        stream_name2 = markers_stream2["info"]["name"][0]
        time_stamps2 = markers_stream2["time_stamps"]
        time_series2 = markers_stream2["time_series"]
        n_markers2 = len(time_stamps2)

        # ensure the time series contains strings
        if not isinstance(time_series2[0], str):
            time_series2 = [str(x) for x in time_series2]

        # ensure the time series is a numpy array
        if isinstance(time_series2, list):
            time_series2 = np.array(time_series2)

        time_series2 = stream_name2 + " " + np.array(time_series2)

        if title_txt == "":
            title_txt = f"{stream_name1} vs {stream_name2} ({n_markers1} vs {n_markers2} markers)"

        time_stamps = np.concatenate((time_stamps1, time_stamps2))
        time_series = np.concatenate((time_series1, time_series2))

        timestamp_marker_list = list(zip(time_stamps, time_series))

        # sort the markers by time
        timestamp_marker_list = sorted(timestamp_marker_list, key=lambda x: x[0])

        plot_markers_csv_time_difference(timestamp_marker_list, title_txt=title_txt)

    if doRunTests:
        print_streams_types_and_names(xdf_data)

        if nic_markers:
            print_and_plot_markers_stream(nic_markers)

        if mouse_markers:
            print_and_plot_markers_stream(mouse_markers)

        if mouse_to_nic_markers:
            print_and_plot_markers_stream(mouse_to_nic_markers)

        if condition_reaching_task:
            print_and_plot_markers_stream(condition_reaching_task)

        if event_ide_TONIC:
            print_and_plot_markers_stream(event_ide_TONIC)

        if event_ide_TONIC and condition_reaching_task:
            plot_two_marker_lists(event_ide_TONIC, condition_reaching_task)

        if mouse_markers and mouse_to_nic_markers:
            plot_two_marker_lists(mouse_markers, mouse_to_nic_markers)

        if event_ide_TONIC and nic_markers:
            plot_two_marker_lists(event_ide_TONIC, nic_markers)

    ################################################################################################
    # get the data from the xdf file
    error_msg = ""
    # get the  mouse data (3 streams are expected)
    if not mouse_mocap or not mouse_markers or not mouse_to_nic_markers:
        error_msg += "No Mouse data in the xdf file.\n"
    else:
        mouse_t = mouse_mocap["time_stamps"]
        mouse_x = mouse_mocap["time_series"][:, 0]
        mouse_y = mouse_mocap["time_series"][:, 1]

        # get the start and stop of the mouse motion from the mouse markers
        mouse_markers_data = mouse_markers["time_series"]
        mouse_markers_time = mouse_markers["time_stamps"]

        start_marker_index_list = find__mouse_marker_indexes(
            "DoCycleChange:DoRecord", mouse_markers_data
        )
        stop_marker_index_list = find__mouse_marker_indexes(
            "DoCycleChange:DoPause", mouse_markers_data
        )
        start_times = mouse_markers_time[start_marker_index_list]
        stop_times = mouse_markers_time[stop_marker_index_list]

        # get the start and stop of the mouse motion from MouseToNIC
        mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
        mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]

        if not isinstance(mouse_to_nic_markers_data[0], list):
            mouse_to_nic_markers_data = [[str(x)] for x in mouse_to_nic_markers_data]

        start_marker_index_list = find__mouse_marker_indexes(
            "[100]", mouse_to_nic_markers_data
        )
        stop_marker_index_list = find__mouse_marker_indexes(
            "[111]", mouse_to_nic_markers_data
        )
        start_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
            start_marker_index_list
        ]
        stop_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
            stop_marker_index_list
        ]

    if not nic_markers:
        error_msg += "No NIC-Markers data in the xdf file.\n"
    else:
        # get the start and stop of the mouse motion from NIC-Markers
        nic_markers_data = nic_markers["time_series"]
        nic_markers_time = nic_markers["time_stamps"]

        # convert the markers data to list of list of strings (as in the str markers in xdf)
        if not isinstance(nic_markers_data[0], list):
            nic_markers_data = [[str(x)] for x in nic_markers_data]

        start_marker_index_list = find__mouse_marker_indexes("[25]", nic_markers_data)
        stop_marker_index_list = find__mouse_marker_indexes("[50]", nic_markers_data)

        # start_marker_index_list = find__mouse_marker_indexes(
        #     "[100]", nic_markers_data
        # )
        # stop_marker_index_list = find__mouse_marker_indexes(
        #     "[101]", nic_markers_data
        # )

        start_times_from_nic_markers = nic_markers_time[start_marker_index_list]
        stop_times_from_nic_markers = nic_markers_time[stop_marker_index_list]

    # get the accelerometer data
    if not acc_mocap:
        error_msg += "No Accelerometer data in the xdf file.\n"
    else:
        acc_t = acc_mocap["time_stamps"]
        acc_x = acc_mocap["time_series"][:, 0]
        acc_y = acc_mocap["time_series"][:, 1]
        acc_z = acc_mocap["time_series"][:, 2]

    # get the kinect data
    if not kinect_mocap:
        error_msg += "No Kinect data in the xdf file.\n"
    else:
        kinect_t = kinect_mocap["time_stamps"]
        WristRight_X = get_kinect_channel(kinect_mocap, "WristRight_X")
        WristRight_Y = get_kinect_channel(kinect_mocap, "WristRight_Y")
        WristRight_Z = get_kinect_channel(kinect_mocap, "WristRight_Z")

        WristLeft_X = get_kinect_channel(kinect_mocap, "WristLeft_X")
        WristLeft_Y = get_kinect_channel(kinect_mocap, "WristLeft_Y")
        WristLeft_Z = get_kinect_channel(kinect_mocap, "WristLeft_Z")

        WristLeft_Norm = np.sqrt(WristLeft_X**2 + WristLeft_Y**2 + WristLeft_Z**2)
        WristRight_Norm = np.sqrt(WristRight_X**2 + WristRight_Y**2 + WristRight_Z**2)

        # low pass the Wrist timeseries
        fs = 30
        cutoff = 1

        # we need a minimal number of points to apply the filter
        if kinect_t.size < 10:
            error_msg += "More than 10 points needed to low pass filter kinect data.\n"
        else:
            WristRight_X = low_pass_butt(WristRight_X, cutoff, fs)
            WristRight_Y = low_pass_butt(WristRight_Y, cutoff, fs)
            WristRight_Z = low_pass_butt(WristRight_Z, cutoff, fs)
            WristRight_Norm = low_pass_butt(WristRight_Norm, cutoff, fs)

            WristLeft_Xf = low_pass_butt(WristLeft_X, cutoff, fs)
            WristLeft_Yf = low_pass_butt(WristLeft_Y, cutoff, fs)
            WristLeft_Zf = low_pass_butt(WristLeft_Z, cutoff, fs)
            WristLeft_Norm = low_pass_butt(WristLeft_Norm, cutoff, fs)

        # check if the kinect timestamps need to be corrected
        needs_correction = needs_kinect_timestamps_correction(
            xdf_fullFname, "do_not_debug"
        )
        if needs_correction:
            msg = "Kinect timestamps need to be corrected"
            # correct the xdf_kinect_mocap_time
            delays_for_xdf = get_kinect_to_mouse_delay_for_xdf(xdf_fullFname)
            kinect_to_mouse_delay_mean = delays_for_xdf["kinect_to_mouse_delay_mean"]
            print(f"kinect_to_mouse_delay_mean = {kinect_to_mouse_delay_mean:.6f} s")
            kinect_t_correct = kinect_t + kinect_to_mouse_delay_mean
        else:
            msg = "Kinect timestamps do not need to be corrected"
            kinect_t_correct = kinect_t

    # choose the start_times and stop_times for the markers to plot the start and stop times
    if mouse_to_nic_markers:
        start_times = start_times_from_mouse_to_nic_markers
        stop_times = stop_times_from_mouse_to_nic_markers

    if nic_markers and event_ide_TONIC:
        start_times = start_times_from_nic_markers
        stop_times = stop_times_from_nic_markers

    ################################################################################################
    # make a plot of the kinect and mouse data with the start and stop times
    ################################################################################################

    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
    fig.set_size_inches(11.7, 8.27)  # A4 format

    if kinect_mocap:
        # plot the norm of the right wrist
        ax1.plot(
            kinect_t_correct,
            WristRight_Norm,
            ".",
            markersize=0.5,
            label="right wrist",
            color="black",
        )
        # plot the norm of the left wrist
        ax1.plot(
            kinect_t_correct,
            WristLeft_Norm,
            ".",
            markersize=0.5,
            label="left wrist",
            color="blue",
        )
        # add the y-axis label
        ax1.set_ylabel("Distance to Kinect (m)")
    else:
        # plot the error message
        ax1.text(
            0.5,
            0.5,
            error_msg,
            horizontalalignment="center",
            verticalalignment="center",
            transform=ax1.transAxes,
        )

    if kinect_mocap and mouse_mocap:

        # replot with larger line width from start to stop
        for t_start, t_stop in zip(start_times, stop_times):
            i = np.where((kinect_t_correct >= t_start) & (kinect_t_correct <= t_stop))
            wr = WristRight_Norm[i]
            wl = WristLeft_Norm[i]
            t = kinect_t_correct[i]
            ax1.plot(t, wr, ".", color="black", markersize=3)
            ax1.plot(t, wl, ".", color="blue", markersize=3)
        #  plot the start and stop times as vertical lines
        for t_start in start_times:
            ax1.axvline(x=t_start, color="red", linestyle="--", zorder=200)
            ax2.axvline(x=t_start, color="red", linestyle="--", zorder=200)
    else:
        pass

    def plot_one_marker(
        ax,
        timestamp,
        marker=None,
        line_color="green",
        marker_color="green",
        marker_h_align="center",
    ):
        # plot the vertical line
        ax.axvline(x=timestamp, color=line_color, linestyle="--")
        # add the corresponding marker label
        if marker:
            ax.text(
                timestamp,
                0.5,  # y position = middle of the plot
                marker,
                horizontalalignment=marker_h_align,
                verticalalignment="center",
                transform=ax.get_xaxis_transform(),  # x-axis in data coordinates, y-axis in axes coordinates
                color=marker_color,
                backgroundcolor="white",
                rotation=90,
            )

    def plot_markers_list(
        ax,
        timestamps,
        markers,
        line_color="green",
        marker_color="green",
        marker_h_align="center",
    ):
        for i in range(len(markers)):
            plot_one_marker(
                ax,
                timestamps[i],
                markers[i],
                line_color=line_color,
                marker_color=marker_color,
                marker_h_align=marker_h_align,
            )

    def rename_mouse_to_NIC_markers(markers_list):
        """Rename the markers of the mouse_to_NIC stream"""

        mouse_to_NIC_markers_labels = {
            1: "[ Start trial ]",
            2: "[ End trial ]",
            3: "[ Start task ]",
            4: "[ End task ]",
        }

        # make a copy of the markers list (to avoid modifying the original list)
        markers_txt = markers_list.copy().astype(str)

        for i, item in enumerate(markers_list):
            key = item[0]
            if key in mouse_to_NIC_markers_labels:
                markers_txt[i] = mouse_to_NIC_markers_labels[key]
            else:
                markers_txt[i] = f"[ {key} ]"

        return markers_txt

    def rename_event_ide_ToNIC_markers(markers_list):
        """Rename the markers of the event_ide_TONIC stream"""

        event_ide_ToNIC_markers_labels = {
            25: "[  GO  ]",
            50: "[ STOP ]",
            75: "[ Rest ]",
            100: "[ Beg trial ]",
            101: "[ End trial ]",
            1: "[ paretic spontaneous]",
            2: "[ non-paretic spontaneous ]",
            3: "[ paretic maximal ]",
            4: "[ non-paretic maximal ]",
            155: "[ End task ]",
        }

        # make a copy of the markers list (to avoid modifying the original list)
        markers_txt = markers_list.copy().astype(str)

        for i, item in enumerate(markers_list):
            key = item[0]
            if key in event_ide_ToNIC_markers_labels:
                markers_txt[i] = event_ide_ToNIC_markers_labels[key]
            else:
                markers_txt[i] = f"[ {key} ]"

        return markers_txt

    if kinect_mocap and condition_reaching_task:
        # plot the condition_reaching_task markers
        crt_t = condition_reaching_task["time_stamps"]
        crt_m = condition_reaching_task["time_series"]

        # plot_markers_list(ax1, crt_t, crt_m, marker_h_align="left")
        # plot_markers_list(ax2, crt_t, crt_m, marker_h_align="left")

    if kinect_mocap and event_ide_TONIC:
        # plot the event_ide_TONIC markers
        et_t = event_ide_TONIC["time_stamps"]
        et_m = event_ide_TONIC["time_series"]
        et_m = rename_event_ide_ToNIC_markers(et_m)

        def plot_marker_on_both_axes(
            ax1,
            ax2,
            timestamp,
            marker=None,
            line_color="green",
            marker_color="green",
            marker_h_align="center",
        ):
            if kinect_mocap:
                plot_one_marker(
                    ax1, timestamp, marker, line_color, marker_color, marker_h_align
                )
            if mouse_mocap:
                plot_one_marker(
                    ax2, timestamp, marker, line_color, marker_color, marker_h_align
                )

        # test the plots possibilities
        if 1 == 1:
            beg_plotted = False
            end_plotted = False
            for i in range(len(et_m)):
                timestamp = et_t[i]
                label = et_m[i][0]

                if "?" in label:
                    pass
                elif "GO" in label:
                    # plot_marker_on_both_axes(
                    #     ax1,
                    #     ax2,
                    #     timestamp,
                    #     label,
                    #     line_color="orange",
                    #     marker_color="green",
                    # )
                    pass
                elif "STOP" in label:
                    # plot_marker_on_both_axes(
                    #     ax1,
                    #     ax2,
                    #     timestamp,
                    #     label,
                    #     line_color="orange",
                    #     marker_color="red",
                    # )
                    pass
                elif "Rest" in label and not end_plotted:
                    plot_marker_on_both_axes(
                        ax1,
                        ax2,
                        timestamp,
                        marker=None,  # marker=label
                        line_color="red",
                        marker_color="blue",
                    )
                    end_plotted = True
                    beg_plotted = False

                elif "Beg" in label and not beg_plotted:
                    plot_marker_on_both_axes(
                        ax1,
                        ax2,
                        timestamp,
                        marker=None,
                        line_color="green",
                        marker_color="blue",
                    )
                    beg_plotted = True
                    end_plotted = False

                elif "End trial" in label:
                    # plot_marker_on_both_axes(
                    #     ax1,
                    #     ax2,
                    #     timestamp,
                    #     label,
                    #     line_color="red",
                    #     marker_color="blue",
                    # )
                    pass

                elif "End task" in label:
                    # plot_marker_on_both_axes(
                    #     ax1,
                    #     ax2,
                    #     timestamp,
                    #     marker=None,
                    #     line_color="red",
                    #     marker_color="red",
                    # )
                    pass
                # else: # default
                #     plot_marker_on_both_axes(
                #         ax1,
                #         ax2,
                #         timestamp,
                #         label,
                #         line_color="orange",
                #         marker_color="orange",
                #     )

        else:
            for i in range(len(start_times)):
                timestamp = start_times[i]
                label = "[ Start ]"
                plot_marker_on_both_axes(
                    ax1,
                    ax2,
                    timestamp,
                    label,
                    line_color="orange",
                    marker_color="green",
                    marker_h_align="right",  # before the start time
                )

            for i in range(len(stop_times)):
                timestamp = stop_times[i]
                label = "[ Stop ]"
                plot_marker_on_both_axes(
                    ax1,
                    ax2,
                    timestamp,
                    label,
                    line_color="orange",
                    marker_color="red",
                    marker_h_align="left",  # after the stop time
                )

    if acc_mocap:
        pass

    if mouse_mocap:
        # plot the mouse data
        ax2.plot(mouse_t, mouse_x, ".", markersize=0.5, label="Mouse x", color="orange")
        ax2.plot(mouse_t, mouse_y, ".", markersize=0.5, label="Mouse y", color="green")
        # replot with larger line width from start to stop
        for t_start, t_stop in zip(start_times, stop_times):
            j = np.where((mouse_t >= t_start) & (mouse_t <= t_stop))
            x = mouse_x[j]
            y = mouse_y[j]
            t = mouse_t[j]
            ax2.plot(t, x, ".", markersize=3, color="orange")
            ax2.plot(t, y, ".", markersize=3, color="green")
        # plot the start and stop times as vertical lines
        for t_stop in stop_times:
            ax1.axvline(x=t_stop, color="green", linestyle="--", zorder=200)
            ax2.axvline(x=t_stop, color="green", linestyle="--", zorder=200)
        # add the y-axis label and the x-axis label

    else:
        # plot the error message
        ax2.text(
            0.5,
            0.5,
            error_msg,
            horizontalalignment="center",
            verticalalignment="center",
            transform=ax2.transAxes,
            color="red",
        )

    ax1.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    for text in ax1.get_legend().get_texts():
        # if the text is "right wrist"
        if text.get_text() == "right wrist":
            text.set_color("black")
        if text.get_text() == "left wrist":
            text.set_color("blue")

    ax2.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    for text in ax2.get_legend().get_texts():
        if text.get_text() == "Mouse x":
            text.set_color("orange")
        if text.get_text() == "Mouse y":
            text.set_color("green")

    # the x-axis label is common to both plots
    ax2.set_ylabel("Position (pixel)")
    ax2.set_xlabel("Time (s)")

    # add the filename as a title
    rel_path = os.path.dirname(xdf_fullFname)
    rel_path = os.path.relpath(rel_path, "../dat")
    title_txt = os.path.join(
        rel_path,
        os.path.basename(xdf_fullFname),
    )
    ax1.set_title(title_txt)

    # plot the time starting from the first time stamp
    x_ticks_value = ax1.get_xticks()
    new_x_ticks_value = x_ticks_value - x_ticks_value[0]
    x_ticks_labels = [f"{x:0.0f}" for x in new_x_ticks_value]
    ax1.set_xticklabels(x_ticks_labels)
    ax2.set_xticklabels(x_ticks_labels)

    plt.tight_layout()  # Adjust layout for better spacing

    # we might not want to see the plot in batch run
    if option_str == "show_plot":
        plt.show()


# event_ide_TONIC / MouseToNIC (NIC-Markers)


def test_correct_xdf_kinect_mocap_time():

    # NOTE: only circle task has a real record of the mouse motion
    xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"
    xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Circle/ReArm_C1P02_20210409_V2_c.xdf"
    xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_c.xdf"

    # xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Circular/task-V1_Circle.xdf"

    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V2/Training/task-V2_Reach_training.xdf"  # only one kinect line

    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # no mouse data --> eventIDE
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V2/Reaching/task-V2_Reach.xdf"
    #
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V2/Reaching/task-V2_Reach_2.xdf"
    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V3/Reaching/task-V3_Reach.xdf"  # bugged kinect data

    # This is for a reaching task, hence we expect the mouse to be still... no data (or so) in th mouse stream
    xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3/ReArm_C1P02_20210715_V3_Reaching/ReArm_C1P07_20211116_V3_r.xdf"
    # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210820_V2/ReArm_C1P07_20210820_V2_Reaching/ReArm_C1P07_20210820_V2_r.xdf"
    # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210716_V1/ReArm_C1P07_20210802_V1_Reaching/ReArm_C1P07_20210802_V1_r.xdf"

    # This is a file with kinect fulled with zeros
    # xdf_fullFname = "../dat/ReArm.lnk/C1P42/V2/Circular/task-V2_Circle.xdf"

    # xdf_fullFname = (
    #     "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Circular/task-V1_Circle.xdf"  # new format
    # )

    # xdf_fullFname = (
    #     "../dat/ReArm.lnk/DATA_Named/C1P31/V3/Reaching/task-V3_Reach.xdf"  # new format
    # )

    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V2/Reaching/C1P31_RauMic_20230331_2_r.xdf"  # wrong correction?

    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P45/V2/Reaching/task-V2_Reach.xdf"

    # NOTE: to get the exact name and type of the streams in the xdf file
    # print_streams_names_type(xdf_fullFname)

    # visually check the time correction of the Kinect data
    visually_check_time_correction(xdf_fullFname, "show_plot")

    return


if doRunTests:
    test_correct_xdf_kinect_mocap_time()

In [ ]:
def save_kinect_timestamps_correction(xdf_fullFname):
    """Save the kinect timestamps correction to a csv file"""

    if not needs_kinect_timestamps_correction(xdf_fullFname):
        msg = "    The kinect timestamps DO NOT need to be corrected."
        logging.info(msg)
        print(msg)
        return ""

    delays = get_kinect_to_mouse_delay_for_xdf(xdf_fullFname)
    kinect_timestamps_correction = delays["kinect_to_mouse_delay_mean"]

    fname = os.path.basename(xdf_fullFname)
    fname = fname.replace(".xdf", "_xdf_time_correction.csv")
    xdf_time_correction_fullFname = os.path.join(os.path.dirname(xdf_fullFname), fname)

    # save the kinect timestamps correction
    np.savetxt(
        xdf_time_correction_fullFname,
        [kinect_timestamps_correction],
        delimiter=",",
        header="to add to kinect timestamps (s)",
    )
    msg = f"kinect time correction saved in: {xdf_time_correction_fullFname}"
    logging.info(msg)
    print(msg)

    return xdf_time_correction_fullFname


def save_and_close_figure(xdf_fullFname, output_file_type="png"):
    fig_dir = os.path.dirname(xdf_fullFname)
    # go up in directories until we find the V1 V2 or V3 directory
    while os.path.basename(fig_dir) not in ["V1", "V2", "V3"]:
        fig_dir = os.path.dirname(fig_dir)
    # go up one more directory (to the CxPxx directory)
    fig_dir = os.path.dirname(fig_dir)

    fig_name = os.path.basename(xdf_fullFname)

    if output_file_type == "pdf":
        fig_name_pdf = fig_name.replace(".xdf", "_time_correction.pdf")
        fig_fullFname_pdf = os.path.join(fig_dir, fig_name_pdf)
        plt.savefig(fig_fullFname_pdf, bbox_inches="tight")

    if output_file_type == "png":
        fig_name_png = fig_name.replace(".xdf", "_time_correction.png")
        fig_fullFname_png = os.path.join(fig_dir, fig_name_png)
        plt.savefig(fig_fullFname_png, bbox_inches="tight")

    plt.close()


def save_kinect_timestamps_correction_and_visual_check(xdf_fullFname):
    xdf_time_correction_fullFname = save_kinect_timestamps_correction(xdf_fullFname)
    if xdf_time_correction_fullFname != "":
        kinect_timestamps_correction = np.loadtxt(
            xdf_time_correction_fullFname, delimiter=","
        )
        print(
            f"READ: kinect timestamps correction = {kinect_timestamps_correction:.6f} s"
        )

    visually_check_time_correction(xdf_fullFname, "do_not_show_plot")
    save_and_close_figure(xdf_fullFname)


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"
    xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3/ReArm_C1P02_20210715_V3_Circle/ReArm_C1P02_20210715_V3_c.xdf"
    xdf_fullFname = (
        "../dat/ReArm.lnk/DATA_named/C1P07/V3/Reaching/C1P07_MagPau_20211116_3_r.xdf"
    )
    # xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

    xdf_fullFname = (
        "../dat/ReArm.lnk/DATA_named/C1P01/V1/Circle/001_BenMus_20210202_1_c.xdf"
    )
    xdf_fullFname = (
        "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"
    )

    # xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf" # No marker csv files with '_l_m_' in the name

    # xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P09/V3/Reaching/C1P09_DimRit_20211130_3_r.xdf"  # empty kinect stream

    xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P14/V2/Circular/C1P14_CoqNel_20220128_2_c.xdf"  # old format

    xdf_fullFname = (
        "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Circular/task-V1_Circle.xdf"  # new format
    )

    # xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # no mouse data --> eventIDE

    save_kinect_timestamps_correction_and_visual_check(xdf_fullFname)

## Check trajectory by cycle in the circle task
This is useful to check if the trajectory of the hand movement in the circle task is adequately measured by both the kinect and the mouse.

In [ ]:
def plot_trajectory_by_cycle(xdf_fullFname):
    # TODO: to be implemented correctly
    # plot the trajectory of the mouse for each Record/Pause cycle
    for i, (t_start, t_stop) in enumerate(zip(start_times, stop_times)):
        plt.figure()
        # keep only the data between t_start and t_stop
        i_start_mouse = np.where(mouse_t >= t_start)[0][0]
        i_stop_mouse = np.where(mouse_t <= t_stop)[0][-1]
        m_x_ = mouse_x[i_start_mouse:i_stop_mouse]
        m_y_ = mouse_y[i_start_mouse:i_stop_mouse]
        m_z_ = mouse_z[i_start_mouse:i_stop_mouse]
        # centre the mouse data
        m_x_ = m_x_ - np.mean(m_x_)
        m_y_ = m_y_ - np.mean(m_y_)
        m_z_ = m_z_ - np.mean(m_z_)

        # get the amplitude along the x and y axis
        x_amp = np.max(m_x_) - np.min(m_x_)
        y_amp = np.max(m_y_) - np.min(m_y_)
        print(f"Cycle {i+1} Mouse : x_amp = {x_amp:.3f} px, y_amp = {y_amp:.3f} px")

        i_start_kinect = np.where(kinect_t_correct >= t_start)[0][0]
        i_stop_kinect = np.where(kinect_t_correct <= t_stop)[0][-1]
        W_R_x_ = WristRight_X[i_start_kinect:i_stop_kinect]
        W_R_y_ = WristRight_Y[i_start_kinect:i_stop_kinect]
        W_R_z_ = WristRight_Z[i_start_kinect:i_stop_kinect]
        W_L_x = WristLeft_X[i_start_kinect:i_stop_kinect]
        W_L_y = WristLeft_Y[i_start_kinect:i_stop_kinect]
        W_L_z = WristLeft_Z[i_start_kinect:i_stop_kinect]
        # centre the kinect data
        W_R_x_ = W_R_x_ - np.mean(W_R_x_)
        W_R_y_ = W_R_y_ - np.mean(W_R_y_)
        W_R_z_ = W_R_z_ - np.mean(W_R_z_)
        W_L_x = W_L_x - np.mean(W_L_x)
        W_L_y = W_L_y - np.mean(W_L_y)
        W_L_z = W_L_z - np.mean(W_L_z)

        # get the amplitude along the x and y axis
        x_amp_k = np.max(W_R_x_) - np.min(W_R_x_) * 100  # cm
        y_amp_k = np.max(W_R_y_) - np.min(W_R_y_) * 100  # cm
        z_amp_k = np.max(W_R_z_) - np.min(W_R_z_) * 100  # cm
        k_volume = x_amp_k * y_amp_k * z_amp_k
        print(
            f"Cycle {i+1} Kinect x_amp_k = {x_amp_k:.3f} cm, y_amp_k = {y_amp_k:.3f} cm, z_amp_k = {z_amp_k:.3f} cm, volume = {k_volume:.3f} cm3"
        )

        plt.subplot(131)
        plt.plot(W_R_x_, W_R_z_, ".", markersize=2, label=f"WristRight XZ{len(W_R_z_)}")
        plt.axis("equal")
        plt.title("Wrist XZ")

        plt.subplot(132)
        plt.plot(W_R_y_, W_R_z_, ".", markersize=2, label=f"WristRight YZ{len(W_R_z_)}")
        plt.axis("equal")
        plt.title("Wrist YZ")

        plt.subplot(133)
        plt.plot(m_x_, m_y_, ".", markersize=2, label=f"Mouse{len(m_x_)}")
        plt.axis("equal")
        plt.title(f"Mouse {i+1}")
        plt.legend()

        plt.show()

## Compute the kinect-to-mouse delay for all xdf files in the visit directory



In [ ]:
def get_xdf_files_in_visit(visit_dir, directories_to_skip=None):
    """Get the xdf files in the visit_dir"""

    xdf_files = []

    if not os.path.exists(visit_dir):
        raise ValueError(f"Directory {visit_dir} does not exist")

    for root, dirs, files in os.walk(visit_dir):
        # Skip the directories that are in the directories_to_skip list
        if directories_to_skip and any(
            skip_dir in root for skip_dir in directories_to_skip
        ):
            continue
        for file in files:
            if directories_to_skip and any(
                skip_dir in root for skip_dir in directories_to_skip
            ):
                continue
            if file.endswith(".xdf"):
                xdf_files.append(os.path.join(root, file))

    if not xdf_files:
        logging.warning(f"No xdf files found in {visit_dir}")

    return xdf_files


def is_already_done_kinect_time_correction_in_visit(visitPath, checkLog_fname):
    """
    Check if the visit was already processed with kinect_time_correction
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)
    return os.path.isfile(full_checkLog_fname)


def create_log_file(visitPath, checkLog_fname):
    """
    Create the log file
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)

    # Create the log file
    logging.basicConfig(
        filename=full_checkLog_fname,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True,  # remove previous handlers and set the new one
    )

    return full_checkLog_fname


def merge_png_files_to_pdf(visit_path):
    """
    Merge the png files in the visit folder
    """
    png_files = [
        f for f in os.listdir(visit_path) if f.endswith("_time_correction.png")
    ]
    png_files.sort()

    if len(png_files) > 1:
        # read the png files
        images = [
            Image.open(os.path.join(visit_path, png_file)) for png_file in png_files
        ]
        # convert to RGB
        images = [img.convert("RGB") for img in images]

        images[0].save(
            os.path.join(
                visit_path, f"{os.path.basename(visit_path)}_time_corrections_png.pdf"
            ),
            save_all=True,
            append_images=images[1:],
        )

        # remove the original png files
        for png_file in png_files:
            os.remove(os.path.join(visit_path, png_file))
            # print(f"    Removed {png_file}")
    else:
        msg = f"    No png files to merge in {visit_path}"
        logging.info(msg)
        print(msg)


def merge_pdf_files_to_pdf(visit_path):
    """
    Merge the pdf files in the visit folder
    """
    pdf_files = [
        f for f in os.listdir(visit_path) if f.endswith("_time_correction.pdf")
    ]
    pdf_files.sort()

    from pypdf import PdfWriter

    if len(pdf_files) > 1:
        # read the pdf files
        pdf_merger = PdfWriter()
        for pdf_file in pdf_files:
            pdf_merger.append(os.path.join(visit_path, pdf_file))

        pdf_merger.write(
            os.path.join(
                visit_path, f"{os.path.basename(visit_path)}_time_corrections_pdf.pdf"
            )
        )
        pdf_merger.close()

        # remove the original pdf files
        for pdf_file in pdf_files:
            os.remove(os.path.join(visit_path, pdf_file))
            # print(f"    Removed {pdf_file}")
            pass
    else:
        msg = f"    No pdf files to merge in {visit_path}"
        logging.info(msg)
        print(msg)


def correct_xdf_kinect_timestamps(visit_dir, directories_to_skip=None):
    """Correct the kinect timestamps for all the xdf files in the visit_dir"""

    checkLog_fname = "kinect_time_correction.log"
    if is_already_done_kinect_time_correction_in_visit(visit_dir, checkLog_fname):
        print(f"    Already done: '{checkLog_fname}' found")
        return

    full_checkLog_fname = create_log_file(visit_dir, checkLog_fname)

    logging.info(f"Starting kinect time correction in {visit_dir}")

    xdf_files = get_xdf_files_in_visit(visit_dir, directories_to_skip)

    for xdf_fullFname in xdf_files:
        print(f"---- \n{xdf_fullFname}")
        # if "old" in xdf_fullFname:
        #     print("skip old files")
        #     continue
        logging.info(f" {os.path.basename(xdf_fullFname)}")
        save_kinect_timestamps_correction_and_visual_check(xdf_fullFname)

    logging.info("kinect time correction completed")
    print(f"    Kinect time correction completed: see '{checkLog_fname}' for details")


if doRunTests:
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3"

    visit_dir = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1"
    visit_dir = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V2"
    visit_dir = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V3"

    visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210716_V1"
    # visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210820_V2"
    # visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3"

    visit_dir = "../dat/ReArm.lnk/C1P42/V1"
    visit_dir = "../dat/ReArm.lnk/C1P42/V2"
    visit_dir = "../dat/ReArm.lnk/C1P42/V3"

    visit_dir = (
        "../dat/ReArm.lnk/DATA_named/C1P02/V2"  # /Armeo/002_CorJea_20210409_2_a.xdf"
    )

    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P01/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P21/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P23/V1"
    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P20/V2"

    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P38/V1"

    correct_xdf_kinect_timestamps(
        visit_dir, directories_to_skip=["old", "Training", "Armeo"]
    )
    merge_png_files_to_pdf(os.path.dirname(visit_dir))